# Video Game Sales — Data Pipeline

**Dataset:** Video Game Sales (`vgsales.csv`)  
**Source:** https://www.kaggle.com/datasets/gregorut/videogamesales  

### About the Dataset
This dataset was scraped from VGChartz and contains sales records for video games that sold more than 100,000 copies. Each row represents a single game on a specific platform and includes the game title, platform, release year, genre, publisher, and sales figures broken down by region (North America, Europe, Japan, and Other), plus a global total (in millions of units).

### Analysis Intent
- Which **platforms and genres** generate the most global sales?
- How do **regional preferences** differ between NA, EU, and JP markets?
- What does the **sales trend over time** look like across the industry?
- Which **publishers** dominate by title count vs. by total sales?

---
## 1. Data Acquisition / Loading

**Reproducible download steps (Kaggle CLI):**
```bash
pip install kaggle
kaggle datasets download -d gregorut/videogamesales
unzip videogamesales.zip
```
The file `vgsales.csv` is then used as-is (raw, unmodified).

In [31]:
import pandas as pd
import sqlite3


RAW_PATH     = 'vgsales.csv'
CLEANED_PATH = 'vgsales_cleaned.csv'
PARQUET_PATH = 'vgsales_cleaned.parquet'
DB_PATH      = 'vgsales.db'

# Load raw data — never modified
df_raw = pd.read_csv(RAW_PATH)

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

Shape: 16,598 rows × 11 columns


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [32]:
df_raw.dtypes

Rank              int64
Name             object
Platform         object
Year            float64
Genre            object
Publisher        object
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

In [33]:
df_raw.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


---
## 2. ETL / Cleaning

All cleaning is done on a **copy** of the raw data. `df_raw` remains untouched throughout.

### 2a. Missing Values

In [4]:
df = df_raw.copy()

missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

Missing values per column:
Year         271
Publisher     58
dtype: int64


**Decision:**
- `Year` — rows with missing Year are **dropped**. A release year cannot be meaningfully imputed and is essential for time-series analysis.
- `Publisher` — filled with `'Unknown'`. The game record is still valid for sales and genre analysis even without a publisher name.

In [5]:
year_missing  = df['Year'].isna().sum()
pub_missing   = df['Publisher'].isna().sum()

df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')

print(f'Dropped {year_missing} rows with missing Year')
print(f'Filled  {pub_missing} missing Publisher values → "Unknown"')

Dropped 271 rows with missing Year
Filled  58 missing Publisher values → "Unknown"


### 2b. Data Types

In [6]:
# Year was float due to NaNs — convert to nullable integer
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')

# Enforce float on all sales columns
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
df[sales_cols] = df[sales_cols].apply(pd.to_numeric, errors='coerce')

print('Updated dtypes:')
print(df.dtypes)

Updated dtypes:
Rank              int64
Name             object
Platform         object
Year              Int64
Genre            object
Publisher        object
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object


### 2c. Duplicates

In [7]:
dupes = df.duplicated(subset=['Name', 'Platform', 'Year']).sum()
df = df.drop_duplicates(subset=['Name', 'Platform', 'Year'])
print(f'Removed {dupes} duplicate rows (same Name + Platform + Year)')

Removed 1 duplicate rows (same Name + Platform + Year)


### 2d. Inconsistent Formats

In [8]:
df['Name']      = df['Name'].str.strip()
df['Platform']  = df['Platform'].str.strip().str.upper()
df['Genre']     = df['Genre'].str.strip().str.title()
df['Publisher'] = df['Publisher'].str.strip()

print('Sample after normalisation:')
df[['Name', 'Platform', 'Genre', 'Publisher']].head()

Sample after normalisation:


,Name,Platform,Genre,Publisher
0,Wii Sports,WII,Sports,Nintendo
1,Super Mario Bros.,NES,Platform,Nintendo
2,Mario Kart Wii,WII,Racing,Nintendo
3,Wii Sports Resort,WII,Sports,Nintendo
4,Pokemon Red/Pokemon Blue,GB,Role-Playing,Nintendo


### 2e. Outliers (IQR Method on Global_Sales)

In [9]:
Q1  = df['Global_Sales'].quantile(0.25)
Q3  = df['Global_Sales'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 3 * IQR   # 3×IQR = very permissive fence

outliers = df[df['Global_Sales'] > upper_fence]
print(f'Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}  Upper fence={upper_fence:.2f}')
print(f'{len(outliers)} extreme outliers detected — RETAINED (genuine mega-hits)\n')
outliers[['Name', 'Platform', 'Year', 'Global_Sales']].head(10)

Q1=0.06  Q3=0.48  IQR=0.42  Upper fence=1.74
1004 extreme outliers detected — RETAINED (genuine mega-hits)



,Name,Platform,Year,Global_Sales
0,Wii Sports,WII,2006,82.74
1,Super Mario Bros.,NES,1985,40.24
2,Mario Kart Wii,WII,2008,35.82
3,Wii Sports Resort,WII,2009,33.00
4,Pokemon Red/Pokemon Blue,GB,1996,31.37
5,Tetris,GB,1989,30.26
6,New Super Mario Bros.,DS,2006,30.01
7,Wii Play,WII,2006,29.02
8,New Super Mario Bros. Wii,WII,2009,28.62
9,Duck Hunt,NES,1984,28.31


### 2f. Implausible Year Values

In [10]:
bad = df[(df['Year'] < 1970) | (df['Year'] > 2030)]
print(f'Rows with Year outside 1970–2030: {len(bad)}')
df = df[(df['Year'] >= 1970) & (df['Year'] <= 2030)].reset_index(drop=True)
print(f'\nFinal cleaned shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'(Raw had {df_raw.shape[0]:,} rows)')

Rows with Year outside 1970–2030: 0

Final cleaned shape: 16,326 rows × 11 columns
(Raw had 16,598 rows)


### 2g. Data Dictionary

In [11]:
data_dictionary = pd.DataFrame([
    {'Field': 'Rank',        'Type': 'int',            'Description': 'Original VGChartz sales rank',                           'Transformations': 'Kept as-is; row order may differ after cleaning'},
    {'Field': 'Name',        'Type': 'str',            'Description': 'Title of the video game',                                'Transformations': 'Whitespace stripped'},
    {'Field': 'Platform',    'Type': 'str',            'Description': 'Console/platform abbreviation (e.g. PS4, WII)',           'Transformations': 'Stripped + UPPER CASE'},
    {'Field': 'Year',        'Type': 'Int64 (nullable)','Description': 'Year the game was released',                            'Transformations': 'Parsed to Int64; rows with NaN dropped; values outside 1970-2030 removed'},
    {'Field': 'Genre',       'Type': 'str',            'Description': 'Game genre (e.g. Action, Sports, Racing)',               'Transformations': 'Stripped + Title Cased'},
    {'Field': 'Publisher',   'Type': 'str',            'Description': 'Publishing company',                                     'Transformations': 'NaNs filled with "Unknown"; whitespace stripped'},
    {'Field': 'NA_Sales',    'Type': 'float',          'Description': 'Sales in North America (millions of units)',             'Transformations': 'Coerced to float'},
    {'Field': 'EU_Sales',    'Type': 'float',          'Description': 'Sales in Europe (millions of units)',                    'Transformations': 'Coerced to float'},
    {'Field': 'JP_Sales',    'Type': 'float',          'Description': 'Sales in Japan (millions of units)',                     'Transformations': 'Coerced to float'},
    {'Field': 'Other_Sales', 'Type': 'float',          'Description': 'Sales in all other regions (millions of units)',         'Transformations': 'Coerced to float'},
    {'Field': 'Global_Sales','Type': 'float',          'Description': 'Total worldwide sales (millions); sum of regional cols', 'Transformations': 'Coerced to float; outliers flagged and retained'},
])

data_dictionary

,Field,Type,Description,Transformations
0,Rank,int,Original VGChartz sales rank,Kept as-is; row order may differ after cleaning
1,Name,str,Title of the video game,Whitespace stripped
2,Platform,str,"Console/platform abbreviation (e.g. PS4, WII)",Stripped + UPPER CASE
3,Year,Int64 (nullable),Year the game was released,Parsed to Int64; rows with NaN dropped; values...
4,Genre,str,"Game genre (e.g. Action, Sports, Racing)",Stripped + Title Cased
5,Publisher,str,Publishing company,"NaNs filled with ""Unknown""; whitespace stripped"
6,NA_Sales,float,Sales in North America (millions of units),Coerced to float
7,EU_Sales,float,Sales in Europe (millions of units),Coerced to float
8,JP_Sales,float,Sales in Japan (millions of units),Coerced to float
9,Other_Sales,float,Sales in all other regions (millions of units),Coerced to float


---
## 3. Storage of Cleaned Data

In [20]:
# 3a — Cleaned CSV
df.to_csv(CLEANED_PATH, index=False)
print(f'✓ Cleaned CSV saved  → {CLEANED_PATH}')

# 3b — SQLite database
conn = sqlite3.connect(DB_PATH)
df.to_sql('games', conn, if_exists='replace', index=False)
data_dictionary.to_sql('data_dictionary', conn, if_exists='replace', index=False)
print(f'✓ SQLite DB created  → {DB_PATH}  (tables: games, data_dictionary)')

✓ Cleaned CSV saved  → vgsales_cleaned.csv
✓ SQLite DB created  → vgsales.db  (tables: games, data_dictionary)


---
## 4. Queryability Demonstration (SQL)

All queries run against the SQLite database to demonstrate the data is loadable and queryable.

In [21]:
# Top 5 platforms by global sales
pd.read_sql_query('''
    SELECT Platform, ROUND(SUM(Global_Sales), 2) AS Total_Global_Sales
    FROM games
    GROUP BY Platform
    ORDER BY Total_Global_Sales DESC
    LIMIT 5
''', conn)

,Platform,Total_Global_Sales
0,PS2,1233.46
1,X360,969.61
2,PS3,949.34
3,WII,909.81
4,DS,818.96


In [22]:
# Top 5 genres by global sales
pd.read_sql_query('''
    SELECT Genre, ROUND(SUM(Global_Sales), 2) AS Total_Global_Sales
    FROM games
    GROUP BY Genre
    ORDER BY Total_Global_Sales DESC
    LIMIT 5
''', conn)

,Genre,Total_Global_Sales
0,Action,1722.88
1,Sports,1309.23
2,Shooter,1026.20
3,Role-Playing,923.84
4,Platform,829.15


In [23]:
# Top 5 publishers by number of titles
pd.read_sql_query('''
    SELECT Publisher, COUNT(*) AS Num_Titles
    FROM games
    GROUP BY Publisher
    ORDER BY Num_Titles DESC
    LIMIT 5
''', conn)

,Publisher,Num_Titles
0,Electronic Arts,1338
1,Activision,966
2,Namco Bandai Games,928
3,Ubisoft,918
4,Konami Digital Entertainment,823


In [24]:
# Annual global sales trend 2000–2015
pd.read_sql_query('''
    SELECT Year, ROUND(SUM(Global_Sales), 2) AS Annual_Sales
    FROM games
    WHERE Year BETWEEN 2000 AND 2015
    GROUP BY Year
    ORDER BY Year
''', conn)

,Year,Annual_Sales
0,2000,201.56
1,2001,331.47
2,2002,395.52
3,2003,357.85
4,2004,419.31
5,2005,459.94
6,2006,521.04
7,2007,611.13
8,2008,678.90
9,2009,667.30


In [25]:
# Regional sales breakdown (totals)
pd.read_sql_query('''
    SELECT
        ROUND(SUM(NA_Sales),    2) AS NA_Total,
        ROUND(SUM(EU_Sales),    2) AS EU_Total,
        ROUND(SUM(JP_Sales),    2) AS JP_Total,
        ROUND(SUM(Other_Sales), 2) AS Other_Total,
        ROUND(SUM(Global_Sales),2) AS Global_Total
    FROM games
''', conn)

,NA_Total,EU_Total,JP_Total,Other_Total,Global_Total
0,4333.43,2409.11,1284.3,789.01,8820.35


In [26]:
# View the data dictionary stored in DB
pd.read_sql_query('SELECT * FROM data_dictionary', conn)

,Field,Type,Description,Transformations
0,Rank,int,Original VGChartz sales rank,Kept as-is; row order may differ after cleaning
1,Name,str,Title of the video game,Whitespace stripped
2,Platform,str,"Console/platform abbreviation (e.g. PS4, WII)",Stripped + UPPER CASE
3,Year,Int64 (nullable),Year the game was released,Parsed to Int64; rows with NaN dropped; values...
4,Genre,str,"Game genre (e.g. Action, Sports, Racing)",Stripped + Title Cased
5,Publisher,str,Publishing company,"NaNs filled with ""Unknown""; whitespace stripped"
6,NA_Sales,float,Sales in North America (millions of units),Coerced to float
7,EU_Sales,float,Sales in Europe (millions of units),Coerced to float
8,JP_Sales,float,Sales in Japan (millions of units),Coerced to float
9,Other_Sales,float,Sales in all other regions (millions of units),Coerced to float


In [27]:
conn.close()
print('Pipeline complete.')
print(f'  {RAW_PATH}       — original raw data (untouched)')
print(f'  {CLEANED_PATH}   — cleaned CSV')
print(f'  {PARQUET_PATH}   — cleaned Parquet')
print(f'  {DB_PATH}        — SQLite DB (tables: games, data_dictionary)')

Pipeline complete.
  vgsales.csv       — original raw data (untouched)
  vgsales_cleaned.csv   — cleaned CSV
  vgsales_cleaned.parquet   — cleaned Parquet
  vgsales.db        — SQLite DB (tables: games, data_dictionary)
